In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- GPU guard: stop before spending quota if Kaggle gave us the wrong accelerator ---
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names)
if not names or not all("T4" in n for n in names):
    raise SystemExit(f"Expected T4 GPUs, got {names}. Set Accelerator -> 'GPU T4 x2' and run again.")


In [ ]:
# --- Reword ~40% of requests with Qwen2.5-7B-Instruct-AWQ (Apache-2.0); output tagged ai-reworded ---
# vLLM in its own environment: installed into Kaggle's Python it replaced torch with a CUDA 13 build
# that the preinstalled torchaudio rejects. 0.6.6 still supports T4 (Turing) GPUs with AWQ.
get_ipython().system("pip install -q uv && uv python install 3.11 && uv venv -q -p 3.11 /tmp/vllm_env "
                     "&& uv pip install -q -p /tmp/vllm_env/bin/python vllm==0.6.6.post1")
VLLM_PY = "/tmp/vllm_env/bin/python"
parts = [d + "/m1" for d in inputs_with("m1/train_parts", "vishalam-agent-data-p")]
print("agent data parts:", len(parts))
get_ipython().system(f"python -m vishalam.data.agent.build_m1 --out /tmp/m1 --merge {','.join(parts)}")
m1 = "/tmp/m1"
os.makedirs(f"{WORK}/m1", exist_ok=True)
for f in ["test_seen.jsonl", "test_unseen.jsonl", "stats.json"]:
    shutil.copy(f"{m1}/{f}", f"{WORK}/m1/{f}")
get_ipython().system(f"{VLLM_PY} -m vishalam.data.agent.reword_qwen --inp {m1}/train.jsonl "
                     f"--out {WORK}/m1/train_mixed.jsonl --fraction 0.4")  # one T4: tensor parallel over PCIe is a needless risk
